# Paper 1 — CIFAR-10 Matched Hold-out vs Five-Fold OOF Stacking

This final notebook was prepared from the uploaded **Leakage-Free OOF Hybrid Stacking** notebook.

It retains the exact reconstructed CIFAR-10 architectures used to generate the saved OOF results:

- CNN
- VGG16-style network
- ResNet-style network
- EfficientNet-style network

The notebook does **not** rerun completed five-fold OOF training or completed final-model test prediction generation. It trains only the four missing fixed-hold-out models and then compares:

- soft voting,
- fixed-hold-out Logistic Regression,
- fixed-hold-out LightGBM,
- five-fold OOF Logistic Regression,
- five-fold OOF LightGBM.


## Cell 1 — Install dependencies and mount Drive


In [ ]:
!pip -q install lightgbm

from google.colab import drive
drive.mount("/content/drive")


## Cell 2 — Imports, reproducibility and paths


In [ ]:
import os
import gc
import random
import warnings

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models, Model, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    log_loss,
    classification_report
)

from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

# Existing completed outputs
BASE_OUTPUT_DIR = "/content/drive/MyDrive/OOF_Stacking_Results"
OOF_DIR = os.path.join(BASE_OUTPUT_DIR, "OOF_Probabilities")
FINAL_BASE_MODEL_DIR = os.path.join(BASE_OUTPUT_DIR, "Final_Base_Models")

# New matched-protocol outputs
PAPER_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison/CIFAR10"
)

SPLIT_DIR = os.path.join(PAPER_DIR, "Split_Indices")
HOLDOUT_PROB_DIR = os.path.join(PAPER_DIR, "Holdout_Probabilities")
HOLDOUT_MODEL_DIR = os.path.join(PAPER_DIR, "Holdout_Models")
META_MODEL_DIR = os.path.join(PAPER_DIR, "Meta_Models")
PREDICTION_DIR = os.path.join(PAPER_DIR, "Test_Predictions")
RESULT_DIR = os.path.join(PAPER_DIR, "Results")
FIGURE_DIR = os.path.join(PAPER_DIR, "Figures")

for folder in [
    PAPER_DIR,
    SPLIT_DIR,
    HOLDOUT_PROB_DIR,
    HOLDOUT_MODEL_DIR,
    META_MODEL_DIR,
    PREDICTION_DIR,
    RESULT_DIR,
    FIGURE_DIR
]:
    os.makedirs(folder, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Existing OOF folder:", OOF_DIR)
print("New paper folder:", PAPER_DIR)


## Cell 3 — Load CIFAR-10


In [ ]:
# ============================================================
# CELL 3 — LOAD CIFAR-10 FROM GOOGLE DRIVE CACHE
# ============================================================

import os
import numpy as np

CIFAR_CACHE_DIR = (
    "/content/drive/MyDrive/CIFAR10_Cache"
)

train_images_path = os.path.join(
    CIFAR_CACHE_DIR,
    "X_cf_train.npy"
)

train_labels_path = os.path.join(
    CIFAR_CACHE_DIR,
    "y_cf_train.npy"
)

test_images_path = os.path.join(
    CIFAR_CACHE_DIR,
    "X_cf_test.npy"
)

test_labels_path = os.path.join(
    CIFAR_CACHE_DIR,
    "y_cf_test.npy"
)

required_files = [
    train_images_path,
    train_labels_path,
    test_images_path,
    test_labels_path
]

missing_files = [
    path
    for path in required_files
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "The CIFAR-10 cache files were not found:\n"
        + "\n".join(missing_files)
        + "\n\nCheck the CIFAR_CACHE_DIR path."
    )

print("Loading CIFAR-10 from Google Drive...")

X_cf_train_raw = np.load(
    train_images_path
)

y_cf_train = np.load(
    train_labels_path
)

X_cf_test_raw = np.load(
    test_images_path
)

y_cf_test = np.load(
    test_labels_path
)

# Standardize labels
y_cf_train = np.asarray(
    y_cf_train
).reshape(-1).astype(np.int64)

y_cf_test = np.asarray(
    y_cf_test
).reshape(-1).astype(np.int64)

# Preserve raw uint8 arrays
X_cf_train_raw = np.asarray(
    X_cf_train_raw
)

X_cf_test_raw = np.asarray(
    X_cf_test_raw
)

# Convert to normalized float32 for the matched models
X_cf_train = X_cf_train_raw.astype(
    np.float32
)

X_cf_test = X_cf_test_raw.astype(
    np.float32
)

if X_cf_train.max() > 1.0:
    X_cf_train /= 255.0

if X_cf_test.max() > 1.0:
    X_cf_test /= 255.0

CIFAR_CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

# Validation checks
assert X_cf_train.shape == (
    50000,
    32,
    32,
    3
)

assert X_cf_test.shape == (
    10000,
    32,
    32,
    3
)

assert y_cf_train.shape == (50000,)
assert y_cf_test.shape == (10000,)

print("CIFAR-10 loaded successfully from Drive.")
print("Development:", X_cf_train.shape)
print("Official test:", X_cf_test.shape)
print(
    "Development class counts:",
    np.bincount(y_cf_train)
)
print(
    "Test class counts:",
    np.bincount(y_cf_test)
)
print(
    "Training pixel range:",
    float(X_cf_train.min()),
    "to",
    float(X_cf_train.max())
)

## Cell 4 — Create or reload the fixed 80:20 split


In [ ]:
base_index_path = os.path.join(
    SPLIT_DIR,
    "CIFAR10_Base_Train_Indices.npy"
)

meta_index_path = os.path.join(
    SPLIT_DIR,
    "CIFAR10_Meta_Holdout_Indices.npy"
)

all_indices = np.arange(len(y_cf_train))

if (
    os.path.exists(base_index_path)
    and os.path.exists(meta_index_path)
):
    base_indices = np.load(base_index_path)
    meta_indices = np.load(meta_index_path)
    print("Loaded the existing fixed split.")
else:
    base_indices, meta_indices = train_test_split(
        all_indices,
        test_size=0.20,
        stratify=y_cf_train,
        random_state=RANDOM_STATE
    )

    np.save(base_index_path, base_indices)
    np.save(meta_index_path, meta_indices)
    print("Created and saved the fixed split.")

X_base = X_cf_train[base_indices]
y_base = y_cf_train[base_indices]

X_meta_holdout = X_cf_train[meta_indices]
y_meta_holdout = y_cf_train[meta_indices]

print("Base-model subset:", X_base.shape)
print("Meta-training hold-out:", X_meta_holdout.shape)
print("Base class counts:", np.bincount(y_base))
print("Meta class counts:", np.bincount(y_meta_holdout))


## Cell 5 — Exact augmentation layers and model builders

These definitions are copied from the uploaded leakage-free OOF notebook.


In [ ]:
VERIFY_CNN_LEARNING_RATE = 1e-3
VERIFY_VGG16_LEARNING_RATE = 1e-3
VERIFY_RESNET_LEARNING_RATE = 1e-3
VERIFY_EFFICIENTNET_LEARNING_RATE = 1e-3

def make_cifar_augmentation(name):
    return tf.keras.Sequential(
        [
            layers.RandomFlip("horizontal"),
            layers.RandomTranslation(
                height_factor=0.10,
                width_factor=0.10,
                fill_mode="reflect"
            ),
            layers.RandomRotation(
                factor=10 / 360,
                fill_mode="reflect"
            )
        ],
        name=name
    )

cnn_cifar_augmentation = make_cifar_augmentation(
    "cnn_cifar_augmentation"
)

vgg16_cifar_augmentation = make_cifar_augmentation(
    "vgg16_cifar_augmentation"
)

resnet_cifar_augmentation = make_cifar_augmentation(
    "resnet_cifar_augmentation"
)

efficientnet_cifar_augmentation = make_cifar_augmentation(
    "efficientnet_cifar_augmentation"
)


def cifar_residual_block(
    inputs,
    filters,
    stride=1,
    block_name="residual_block"
):
    shortcut = inputs

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=stride,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{block_name}_conv1"
    )(inputs)

    x = layers.BatchNormalization(
        name=f"{block_name}_bn1"
    )(x)

    x = layers.Activation(
        "relu",
        name=f"{block_name}_relu1"
    )(x)

    x = layers.Conv2D(
        filters=filters,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{block_name}_conv2"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_bn2"
    )(x)

    if stride != 1 or int(inputs.shape[-1]) != filters:

        shortcut = layers.Conv2D(
            filters=filters,
            kernel_size=1,
            strides=stride,
            padding="same",
            use_bias=False,
            kernel_initializer="he_normal",
            name=f"{block_name}_shortcut_conv"
        )(shortcut)

        shortcut = layers.BatchNormalization(
            name=f"{block_name}_shortcut_bn"
        )(shortcut)

    x = layers.Add(
        name=f"{block_name}_add"
    )([x, shortcut])

    x = layers.Activation(
        "relu",
        name=f"{block_name}_output"
    )(x)

    return x

def build_cnn_cifar():
    """
    Fully trainable CIFAR-specific CNN.
    """

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_cnn_input"
    )

    x = cnn_cifar_augmentation(inputs)

    # Block 1
    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.20)(x)

    # Block 2
    x = layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.30)(x)

    # Block 3
    x = layers.Conv2D(
        256,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        256,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.40)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_cnn_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_CNN_Reconstructed"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=VERIFY_CNN_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

def build_vgg16_cifar():
    """
    CIFAR-specific VGG16-style model trained from scratch.

    This is not the frozen ImageNet VGG16 model.
    All convolutional blocks are fully trainable.
    """

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_vgg16_input"
    )

    x = vgg16_cifar_augmentation(inputs)

    # --------------------------------------------------------
    # Block 1: 64 filters
    # --------------------------------------------------------

    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.20)(x)

    # --------------------------------------------------------
    # Block 2: 128 filters
    # --------------------------------------------------------

    x = layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        128,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.30)(x)

    # --------------------------------------------------------
    # Block 3: 256 filters
    # --------------------------------------------------------

    x = layers.Conv2D(
        256,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        256,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        256,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Dropout(0.40)(x)

    # --------------------------------------------------------
    # Block 4: 512 filters
    # --------------------------------------------------------

    x = layers.Conv2D(
        512,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        512,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)

    x = layers.Conv2D(
        512,
        kernel_size=3,
        padding="same",
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)

    # --------------------------------------------------------
    # Classification head
    # --------------------------------------------------------

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)

    x = layers.Dense(
        256,
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_vgg16_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_VGG16_Reconstructed"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=VERIFY_VGG16_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

def build_resnet50_cifar():
    """
    CIFAR-specific residual network trained from scratch.

    This does not use frozen ImageNet weights.
    All convolutional layers are trainable.
    """

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_resnet_input"
    )

    x = resnet_cifar_augmentation(inputs)

    # CIFAR stem: 3×3 convolution, no initial max pooling
    x = layers.Conv2D(
        filters=64,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv"
    )(x)

    x = layers.BatchNormalization(
        name="stem_bn"
    )(x)

    x = layers.Activation(
        "relu",
        name="stem_relu"
    )(x)

    # Stage 1
    x = cifar_residual_block(
        x, 64, stride=1,
        block_name="stage1_block1"
    )

    x = cifar_residual_block(
        x, 64, stride=1,
        block_name="stage1_block2"
    )

    x = layers.Dropout(0.20)(x)

    # Stage 2
    x = cifar_residual_block(
        x, 128, stride=2,
        block_name="stage2_block1"
    )

    x = cifar_residual_block(
        x, 128, stride=1,
        block_name="stage2_block2"
    )

    x = layers.Dropout(0.30)(x)

    # Stage 3
    x = cifar_residual_block(
        x, 256, stride=2,
        block_name="stage3_block1"
    )

    x = cifar_residual_block(
        x, 256, stride=1,
        block_name="stage3_block2"
    )

    x = layers.Dropout(0.40)(x)

    # Stage 4
    x = cifar_residual_block(
        x, 512, stride=2,
        block_name="stage4_block1"
    )

    x = cifar_residual_block(
        x, 512, stride=1,
        block_name="stage4_block2"
    )

    x = layers.Dropout(0.40)(x)

    # Classification head
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(
        512,
        activation="relu",
        kernel_initializer="he_normal"
    )(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_resnet_output"
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_ResNet_Reconstructed"
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=VERIFY_RESNET_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

def squeeze_excite_block(
    inputs,
    se_ratio=0.25,
    name="se"
):
    filters = int(inputs.shape[-1])

    reduced_filters = max(
        1,
        int(filters * se_ratio)
    )

    x = layers.GlobalAveragePooling2D(
        keepdims=True,
        name=f"{name}_gap"
    )(inputs)

    x = layers.Conv2D(
        reduced_filters,
        kernel_size=1,
        activation="swish",
        padding="same",
        name=f"{name}_reduce"
    )(x)

    x = layers.Conv2D(
        filters,
        kernel_size=1,
        activation="sigmoid",
        padding="same",
        name=f"{name}_expand"
    )(x)

    return layers.Multiply(
        name=f"{name}_scale"
    )([inputs, x])

def mbconv_block(
    inputs,
    output_filters,
    kernel_size,
    stride,
    expansion_ratio,
    dropout_rate=0.0,
    block_name="mbconv"
):
    input_filters = int(inputs.shape[-1])
    expanded_filters = input_filters * expansion_ratio

    x = inputs

    # Expansion
    if expansion_ratio != 1:
        x = layers.Conv2D(
            expanded_filters,
            kernel_size=1,
            padding="same",
            use_bias=False,
            name=f"{block_name}_expand_conv"
        )(x)

        x = layers.BatchNormalization(
            name=f"{block_name}_expand_bn"
        )(x)

        x = layers.Activation(
            "swish",
            name=f"{block_name}_expand_activation"
        )(x)

    # Depthwise convolution
    x = layers.DepthwiseConv2D(
        kernel_size=kernel_size,
        strides=stride,
        padding="same",
        use_bias=False,
        name=f"{block_name}_depthwise"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_depthwise_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name=f"{block_name}_depthwise_activation"
    )(x)

    # Squeeze-and-excitation
    x = squeeze_excite_block(
        x,
        se_ratio=0.25,
        name=f"{block_name}_se"
    )

    # Projection
    x = layers.Conv2D(
        output_filters,
        kernel_size=1,
        padding="same",
        use_bias=False,
        name=f"{block_name}_project_conv"
    )(x)

    x = layers.BatchNormalization(
        name=f"{block_name}_project_bn"
    )(x)

    # Residual connection
    if (
        stride == 1
        and input_filters == output_filters
    ):
        if dropout_rate > 0:
            x = layers.Dropout(
                dropout_rate,
                name=f"{block_name}_dropout"
            )(x)

        x = layers.Add(
            name=f"{block_name}_add"
        )([inputs, x])

    return x

def build_efficientnet_cifar():
    """
    CIFAR-specific EfficientNet-style model trained from scratch.

    It does not use frozen ImageNet weights.
    All layers are trainable.
    """

    inputs = layers.Input(
        shape=(32, 32, 3),
        name="cifar_efficientnet_input"
    )

    x = efficientnet_cifar_augmentation(inputs)

    # Stem
    x = layers.Conv2D(
        32,
        kernel_size=3,
        strides=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="stem_conv"
    )(x)

    x = layers.BatchNormalization(
        name="stem_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name="stem_activation"
    )(x)

    # MBConv sequence
    x = mbconv_block(
        x,
        output_filters=16,
        kernel_size=3,
        stride=1,
        expansion_ratio=1,
        dropout_rate=0.05,
        block_name="block1"
    )

    x = mbconv_block(
        x,
        output_filters=24,
        kernel_size=3,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.05,
        block_name="block2a"
    )

    x = mbconv_block(
        x,
        output_filters=24,
        kernel_size=3,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.05,
        block_name="block2b"
    )

    x = mbconv_block(
        x,
        output_filters=40,
        kernel_size=5,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.10,
        block_name="block3a"
    )

    x = mbconv_block(
        x,
        output_filters=40,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.10,
        block_name="block3b"
    )

    x = mbconv_block(
        x,
        output_filters=80,
        kernel_size=3,
        stride=2,
        expansion_ratio=6,
        dropout_rate=0.15,
        block_name="block4a"
    )

    x = mbconv_block(
        x,
        output_filters=80,
        kernel_size=3,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.15,
        block_name="block4b"
    )

    x = mbconv_block(
        x,
        output_filters=112,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.20,
        block_name="block5"
    )

    x = mbconv_block(
        x,
        output_filters=192,
        kernel_size=5,
        stride=1,
        expansion_ratio=6,
        dropout_rate=0.20,
        block_name="block6"
    )

    # Head
    x = layers.Conv2D(
        1280,
        kernel_size=1,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name="head_conv"
    )(x)

    x = layers.BatchNormalization(
        name="head_bn"
    )(x)

    x = layers.Activation(
        "swish",
        name="head_activation"
    )(x)

    x = layers.GlobalAveragePooling2D(
        name="global_average_pooling"
    )(x)

    x = layers.Dropout(
        0.50,
        name="head_dropout"
    )(x)

    outputs = layers.Dense(
        10,
        activation="softmax",
        dtype="float32",
        name="cifar_efficientnet_output"
    )(x)

    model = models.Model(
        inputs=inputs,
        outputs=outputs,
        name="CIFAR_EfficientNet_Reconstructed"
    )

    model.compile(
        optimizer=optimizers.Adam(
            learning_rate=VERIFY_EFFICIENTNET_LEARNING_RATE
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


MODEL_BUILDERS = {
    "CNN": build_cnn_cifar,
    "VGG16": build_vgg16_cifar,
    "ResNet50": build_resnet50_cifar,
    "EfficientNetB0": build_efficientnet_cifar
}

for model_name, builder in MODEL_BUILDERS.items():
    tf.keras.backend.clear_session()
    check_model = builder()
    print(
        model_name,
        check_model.name,
        check_model.output_shape,
        f"{check_model.count_params():,} parameters"
    )
    del check_model
    gc.collect()


## Cell 6 — Train only missing fixed-holdout models

The training configuration matches the corrected reconstructed OOF runs:

- maximum 30 epochs,
- batch size 64,
- Adam learning rate 0.001,
- early stopping on validation loss,
- augmentation embedded inside each model.

An internal validation subset is drawn only from the 80% base-model subset.  
The 20% meta-holdout subset remains untouched until probability generation.


In [ ]:
HOLDOUT_MAX_EPOCHS = 30
HOLDOUT_BATCH_SIZE = 64
HOLDOUT_PATIENCE = 5

def generate_or_load_holdout_probabilities(
    model_name,
    model_builder
):
    probability_path = os.path.join(
        HOLDOUT_PROB_DIR,
        f"CIFAR_{model_name}_Holdout_Meta_Probabilities.npy"
    )

    model_path = os.path.join(
        HOLDOUT_MODEL_DIR,
        f"CIFAR_{model_name}_Holdout.keras"
    )

    if os.path.exists(probability_path):
        probabilities = np.load(
            probability_path
        ).astype(np.float32)

        expected_shape = (10000, 10)

        if probabilities.shape != expected_shape:
            raise ValueError(
                f"{model_name}: saved probability shape "
                f"{probabilities.shape}, expected {expected_shape}."
            )

        print(
            f"{model_name}: loaded saved hold-out probabilities."
        )
        return probabilities

    (
        X_train_internal,
        X_val_internal,
        y_train_internal,
        y_val_internal
    ) = train_test_split(
        X_base,
        y_base,
        test_size=0.10,
        stratify=y_base,
        random_state=RANDOM_STATE
    )

    print("\n" + "=" * 78)
    print("TRAINING MISSING HOLD-OUT MODEL:", model_name)
    print("=" * 78)
    print("Internal training:", X_train_internal.shape)
    print("Internal validation:", X_val_internal.shape)
    print("Untouched meta-holdout:", X_meta_holdout.shape)

    tf.keras.backend.clear_session()
    gc.collect()

    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    tf.random.set_seed(RANDOM_STATE)

    model = model_builder()

    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=HOLDOUT_PATIENCE,
            min_delta=0.001,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        )
    ]

    model.fit(
        X_train_internal,
        y_train_internal,
        validation_data=(
            X_val_internal,
            y_val_internal
        ),
        epochs=HOLDOUT_MAX_EPOCHS,
        batch_size=HOLDOUT_BATCH_SIZE,
        callbacks=callbacks,
        shuffle=True,
        verbose=2
    )

    probabilities = model.predict(
        X_meta_holdout,
        batch_size=HOLDOUT_BATCH_SIZE,
        verbose=1
    ).astype(np.float32)

    expected_shape = (10000, 10)

    if probabilities.shape != expected_shape:
        raise ValueError(
            f"{model_name}: obtained {probabilities.shape}, "
            f"expected {expected_shape}."
        )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"{model_name}: probabilities contain NaN or infinity."
        )

    row_sum_error = float(
        np.max(
            np.abs(
                probabilities.sum(axis=1) - 1.0
            )
        )
    )

    if row_sum_error > 1e-3:
        raise ValueError(
            f"{model_name}: maximum row-sum error is "
            f"{row_sum_error:.6f}."
        )

    np.save(probability_path, probabilities)
    model.save(model_path)

    print("Saved probabilities:", probability_path)
    print("Saved model:", model_path)

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return probabilities


## Cell 7 — Generate or reload all four hold-out matrices


In [ ]:
holdout_probabilities = {}

for model_name, builder in MODEL_BUILDERS.items():
    holdout_probabilities[
        model_name
    ] = generate_or_load_holdout_probabilities(
        model_name,
        builder
    )

    print(
        model_name,
        holdout_probabilities[model_name].shape
    )


## Cell 8 — Verify and load all completed OOF and final-test arrays

No model training occurs in this cell.


In [ ]:
required_files = {
    "CNN OOF": (
        "CIFAR_CNN_5Fold_OOF.npy",
        (50000, 10)
    ),
    "VGG16 OOF": (
        "CIFAR_VGG16_5Fold_OOF.npy",
        (50000, 10)
    ),
    "ResNet50 OOF": (
        "CIFAR_ResNet50_5Fold_OOF.npy",
        (50000, 10)
    ),
    "EfficientNetB0 OOF": (
        "CIFAR_EfficientNetB0_5Fold_OOF.npy",
        (50000, 10)
    ),
    "CNN Test": (
        "CIFAR_CNN_Test_Probabilities.npy",
        (10000, 10)
    ),
    "VGG16 Test": (
        "CIFAR_VGG16_Test_Probabilities.npy",
        (10000, 10)
    ),
    "ResNet50 Test": (
        "CIFAR_ResNet50_Test_Probabilities.npy",
        (10000, 10)
    ),
    "EfficientNetB0 Test": (
        "CIFAR_EfficientNetB0_Test_Probabilities.npy",
        (10000, 10)
    )
}

loaded_arrays = {}

for label, (filename, expected_shape) in required_files.items():
    path = os.path.join(OOF_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing required saved file:\n{path}"
        )

    array = np.load(path).astype(np.float32)

    if array.shape != expected_shape:
        raise ValueError(
            f"{label}: obtained {array.shape}, "
            f"expected {expected_shape}."
        )

    if not np.isfinite(array).all():
        raise ValueError(
            f"{label}: contains NaN or infinity."
        )

    row_sum_error = float(
        np.max(
            np.abs(
                array.sum(axis=1) - 1.0
            )
        )
    )

    if row_sum_error > 1e-3:
        raise ValueError(
            f"{label}: maximum row-sum error "
            f"{row_sum_error:.6f}."
        )

    loaded_arrays[label] = array
    print(label, array.shape)

print("\nAll completed OOF and test probability files are valid.")


## Cell 9 — Build the 40-feature matrices


In [ ]:
MODEL_ORDER = [
    "CNN",
    "VGG16",
    "ResNet50",
    "EfficientNetB0"
]

oof_probabilities = {
    "CNN": loaded_arrays["CNN OOF"],
    "VGG16": loaded_arrays["VGG16 OOF"],
    "ResNet50": loaded_arrays["ResNet50 OOF"],
    "EfficientNetB0": loaded_arrays["EfficientNetB0 OOF"]
}

test_probabilities = {
    "CNN": loaded_arrays["CNN Test"],
    "VGG16": loaded_arrays["VGG16 Test"],
    "ResNet50": loaded_arrays["ResNet50 Test"],
    "EfficientNetB0": loaded_arrays["EfficientNetB0 Test"]
}

X_holdout_meta = np.concatenate(
    [
        holdout_probabilities[name]
        for name in MODEL_ORDER
    ],
    axis=1
).astype(np.float32)

y_holdout_meta = y_meta_holdout.copy()

X_oof_meta = np.concatenate(
    [
        oof_probabilities[name]
        for name in MODEL_ORDER
    ],
    axis=1
).astype(np.float32)

y_oof_meta = y_cf_train.copy()

X_test_meta = np.concatenate(
    [
        test_probabilities[name]
        for name in MODEL_ORDER
    ],
    axis=1
).astype(np.float32)

assert X_holdout_meta.shape == (10000, 40)
assert X_oof_meta.shape == (50000, 40)
assert X_test_meta.shape == (10000, 40)

print("Hold-out meta:", X_holdout_meta.shape)
print("OOF meta:", X_oof_meta.shape)
print("Test meta:", X_test_meta.shape)

np.save(
    os.path.join(
        HOLDOUT_PROB_DIR,
        "CIFAR10_Holdout_Meta_40_Features.npy"
    ),
    X_holdout_meta
)

np.save(
    os.path.join(
        HOLDOUT_PROB_DIR,
        "CIFAR10_Holdout_Meta_Labels.npy"
    ),
    y_holdout_meta
)


## Cell 10 — Define identical meta-learners

The settings match the uploaded leakage-free OOF notebook and are applied identically to both protocols.


In [ ]:
def build_logistic_meta_learner():
    return LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        multi_class="auto",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


def build_lightgbm_meta_learner():
    return LGBMClassifier(
        objective="multiclass",
        num_class=10,
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.20,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1
    )


## Cell 11 — Train hold-out and OOF meta-learners


In [ ]:
training_jobs = {
    "Holdout_LogisticRegression": (
        build_logistic_meta_learner(),
        X_holdout_meta,
        y_holdout_meta
    ),
    "Holdout_LightGBM": (
        build_lightgbm_meta_learner(),
        X_holdout_meta,
        y_holdout_meta
    ),
    "OOF_LogisticRegression": (
        build_logistic_meta_learner(),
        X_oof_meta,
        y_oof_meta
    ),
    "OOF_LightGBM": (
        build_lightgbm_meta_learner(),
        X_oof_meta,
        y_oof_meta
    )
}

meta_test_probabilities = {}

for job_name, (
    meta_model,
    X_meta_train,
    y_meta_train
) in training_jobs.items():

    model_path = os.path.join(
        META_MODEL_DIR,
        job_name + ".joblib"
    )

    probability_path = os.path.join(
        PREDICTION_DIR,
        job_name + "_Test_Probabilities.npy"
    )

    if (
        os.path.exists(model_path)
        and os.path.exists(probability_path)
    ):
        meta_model = joblib.load(model_path)
        probabilities = np.load(
            probability_path
        ).astype(np.float32)

        print(job_name, "loaded from saved files.")
    else:
        meta_model.fit(
            X_meta_train,
            y_meta_train
        )

        probabilities = meta_model.predict_proba(
            X_test_meta
        ).astype(np.float32)

        joblib.dump(meta_model, model_path)
        np.save(probability_path, probabilities)

        print(job_name, "trained and saved.")

    meta_test_probabilities[job_name] = probabilities


soft_voting_probabilities = np.mean(
    [
        test_probabilities[name]
        for name in MODEL_ORDER
    ],
    axis=0
).astype(np.float32)

soft_voting_probabilities = (
    soft_voting_probabilities
    / soft_voting_probabilities.sum(
        axis=1,
        keepdims=True
    )
)

np.save(
    os.path.join(
        PREDICTION_DIR,
        "SoftVoting_Test_Probabilities.npy"
    ),
    soft_voting_probabilities
)


## Cell 12 — Evaluate and save the matched comparison


In [ ]:
def evaluate_probabilities(
    method,
    protocol,
    probabilities
):
    predictions = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "Dataset": "CIFAR-10",
        "Protocol": protocol,
        "Method": method,
        "Accuracy": accuracy_score(
            y_cf_test,
            predictions
        ),
        "Macro_Precision": precision_score(
            y_cf_test,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_cf_test,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro_F1": f1_score(
            y_cf_test,
            predictions,
            average="macro",
            zero_division=0
        ),
        "ROC_AUC_OVR": roc_auc_score(
            y_cf_test,
            probabilities,
            multi_class="ovr",
            average="macro"
        ),
        "Log_Loss": log_loss(
            y_cf_test,
            probabilities,
            labels=np.arange(10)
        ),
        "Seed": RANDOM_STATE
    }


rows = [
    evaluate_probabilities(
        "Soft Voting",
        "Common baseline",
        soft_voting_probabilities
    )
]

for job_name, probabilities in meta_test_probabilities.items():
    if job_name.startswith("Holdout_"):
        protocol = "Fixed hold-out"
        method = job_name.replace("Holdout_", "")
    else:
        protocol = "Five-fold OOF"
        method = job_name.replace("OOF_", "")

    rows.append(
        evaluate_probabilities(
            method,
            protocol,
            probabilities
        )
    )

cifar_results = pd.DataFrame(rows)

display_table = cifar_results.copy()

for metric in [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "ROC_AUC_OVR",
    "Log_Loss"
]:
    display_table[metric] = (
        display_table[metric].round(6)
    )

display(display_table)

result_path = os.path.join(
    RESULT_DIR,
    "CIFAR10_Matched_Holdout_vs_OOF_Results.csv"
)

cifar_results.to_csv(
    result_path,
    index=False
)

print("Saved:", result_path)


## Cell 13 — Save predicted labels and print classification reports


In [ ]:
all_outputs = {
    "SoftVoting": soft_voting_probabilities,
    **meta_test_probabilities
}

for method_name, probabilities in all_outputs.items():
    predictions = np.argmax(
        probabilities,
        axis=1
    )

    np.save(
        os.path.join(
            PREDICTION_DIR,
            method_name + "_Predicted_Labels.npy"
        ),
        predictions
    )

    print("\n" + "=" * 78)
    print(method_name)
    print("=" * 78)

    print(
        classification_report(
            y_cf_test,
            predictions,
            target_names=CIFAR_CLASS_NAMES,
            digits=4,
            zero_division=0
        )
    )

np.save(
    os.path.join(
        PREDICTION_DIR,
        "CIFAR10_Test_True_Labels.npy"
    ),
    y_cf_test
)


## Cell 14 — Journal-ready chart with values


In [ ]:
plot_df = cifar_results.copy()

label_map = {
    ("Common baseline", "Soft Voting"): "Soft Voting",
    ("Fixed hold-out", "LogisticRegression"): "Hold-out LR",
    ("Fixed hold-out", "LightGBM"): "Hold-out LGBM",
    ("Five-fold OOF", "LogisticRegression"): "OOF LR",
    ("Five-fold OOF", "LightGBM"): "OOF LGBM"
}

plot_df["Label"] = [
    label_map.get(
        (protocol, method),
        f"{protocol} {method}"
    )
    for protocol, method in zip(
        plot_df["Protocol"],
        plot_df["Method"]
    )
]

plot_df["Accuracy_Percent"] = (
    plot_df["Accuracy"] * 100
)

plt.figure(figsize=(10, 6))

bars = plt.bar(
    plot_df["Label"],
    plot_df["Accuracy_Percent"]
)

plt.ylabel("Accuracy (%)")
plt.xlabel("Ensemble method")
plt.title(
    "CIFAR-10 matched hold-out versus five-fold OOF stacking"
)

for bar, value in zip(
    bars,
    plot_df["Accuracy_Percent"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.08,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )

minimum_value = plot_df["Accuracy_Percent"].min()
maximum_value = plot_df["Accuracy_Percent"].max()

plt.ylim(
    max(0, minimum_value - 2),
    min(100, maximum_value + 1.5)
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

plt.tight_layout()

figure_path = os.path.join(
    FIGURE_DIR,
    "CIFAR10_Matched_Holdout_vs_OOF_Accuracy.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", figure_path)


# End of notebook

Completed OOF folds and completed final base-model prediction generation are intentionally excluded.

On the first run, only four fixed-holdout base models are trained. Their probability files are saved. On later runs, those files and all completed OOF/test arrays are loaded automatically.


In [ ]:
# ============================================================
# SOFTWARE ENVIRONMENT
# ============================================================

import sys
import platform
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import lightgbm
import matplotlib
import statsmodels

print("Python:", sys.version)
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("LightGBM:", lightgbm.__version__)
print("Matplotlib:", matplotlib.__version__)
print("statsmodels:", statsmodels.__version__)